# RAG Pipeline Evaluation Project

# ETL part

load EnterpriseRAG-Bench from Hugging Face

In [1]:
# Step 1: Install required library
# "datasets" is the Hugging Face library used to download public datasets.
!pip install datasets -q

## 1. Load EnterpriseRAG-Bench Dataset
This section installs required libraries and loads the documents/questions datasets from Hugging Face.

In [2]:
# Step 2: Import required libraries

# load_dataset is used to load datasets from Hugging Face.
from datasets import load_dataset

# pandas is used to convert the dataset into table format.
import pandas as pd


# Step 3: Load EnterpriseRAG-Bench dataset from Hugging Face

# This dataset has two important parts:
# 1. documents  -> company-like documents used as RAG knowledge base
# 2. questions  -> test questions with expected answers and expected document IDs

documents_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "documents",
    split="test"
)

questions_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions",
    split="test"
)


# Step 4: Convert Hugging Face dataset into pandas DataFrame
# DataFrame means table format, like Excel/CSV.

documents_df = documents_dataset.to_pandas()
questions_df = questions_dataset.to_pandas()


# Step 5: Check how many rows and columns are present

print("Documents dataset shape:", documents_df.shape)
print("Questions dataset shape:", questions_df.shape)


# Step 6: Show column names
# This helps us understand what fields are available.

print("\nDocuments columns:")
print(documents_df.columns.tolist())

print("\nQuestions columns:")
print(questions_df.columns.tolist())


# Step 7: Preview first few rows

print("\nDocuments preview:")
display(documents_df.head())

print("\nQuestions preview:")
display(questions_df.head())

README.md:   0%|          | 0.00/6.93k [00:00<?, ?B/s]

data/documents/test.parquet: reconstructing file:   0%|          |  0.00B / 1.41GB            

data/documents/test.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

data/questions/test.parquet: reconstructing file:   0%|          |  0.00B /  409kB            

...

Generating test split: 0 examples [00:00, ? examples/s]

Documents dataset shape: (511962, 4)
Questions dataset shape: (500, 7)

Documents columns:
['doc_id', 'source_type', 'title', 'content']

Questions columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Documents preview:


,doc_id,source_type,title,content
0,dsid_e54ef48bae78474684a957cf613d47d5,confluence,Runbook: Deploy / Upgrade / Roll Back perf-can...,## Purpose\nThis runbook describes the operati...
1,dsid_229dd48e9b1d466a81ebaffe3ec84469,confluence,Cross-account GPU burst SLO contract and CI li...,Summary\n\nOverview:\nThis document defines th...
2,dsid_aeb0022d62bc43beb6549ba92e5655eb,confluence,Third-Party & Vendor Coordination Playbook for...,Overview\n\nPurpose:\nThis playbook documents ...
3,dsid_926174fc4900408c89c98abde46b7225,confluence,First Production Launch Checklist (Canonical T...,## Purpose\nThis page defines the **canonical ...
4,dsid_d511c6d8daf94f998ce6a3d97462af2d,confluence,Go-Live Runway and Week-3 Stability Playbook,Overview\n\nThis playbook defines the technica...



Questions preview:


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,[fireflies],In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,[gmail],What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...


## 2. Inspect Dataset Structure
This section checks one sample document and one sample question to understand the dataset fields.

In [3]:
# Step 8: Check one full document row
# This helps us understand what one knowledge-base document looks like.

print("One document example:")
display(documents_df.iloc[0])


# Step 9: Check one full question row
# This helps us understand what one test question looks like.

print("One question example:")
display(questions_df.iloc[0])

One document example:


,0
doc_id,dsid_e54ef48bae78474684a957cf613d47d5
source_type,confluence
title,Runbook: Deploy / Upgrade / Roll Back perf-can...
content,## Purpose\nThis runbook describes the operati...


One question example:


,0
question_id,qst_0001
question_type,basic
source_types,[github]
question,What are the default size limits for file uplo...
expected_doc_ids,[dsid_ae068ee4aa9640159427cd941bef0238]
gold_answer,The default limits are 10 MiB per file (max_fi...
answer_facts,[The default per file upload size limit (max_f...


## 3. Select Lightweight QA Test Questions
This section selects 10 basic questions with one clear expected source document.

In [4]:
# Step 10: Select simple/basic questions for our first QA test set

# We are starting with "basic" questions because they are easier to verify.
# Later, we can test complex question types.

basic_questions_df = questions_df[questions_df["question_type"] == "basic"].copy()


# Step 11: Keep only questions that have exactly one expected document
# This makes the first version simple:
# one question -> one correct source document -> one expected answer.

basic_questions_df["expected_doc_count"] = basic_questions_df["expected_doc_ids"].apply(len)

single_doc_questions_df = basic_questions_df[
    basic_questions_df["expected_doc_count"] == 1
].copy()


# Step 12: Select first 10 questions
# We are choosing only 10 because this is the first QA prototype.
# After this works, we can increase the count.

selected_questions_df = single_doc_questions_df.head(10).copy()


# Step 13: Show selected questions

print("Selected questions count:", len(selected_questions_df))

display(
    selected_questions_df[
        [
            "question_id",
            "question_type",
            "question",
            "expected_doc_ids",
            "gold_answer",
            "answer_facts"
        ]
    ]
)

Selected questions count: 10


,question_id,question_type,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...
5,qst_0006,basic,In the draft spec about extending a routing po...,[dsid_184be937d34a412ab5e61366d54d8ed6],The draft proposes this canonical v1 signal pr...,[The draft spec proposes a canonical v1 priori...
6,qst_0007,basic,In a rolling investigation of a model regressi...,[dsid_72ec4a9962ba43e88acd61abbba1052d],"In the first comparison run, the baseline buil...","[In the first comparison run, the older baseli..."
7,qst_0008,basic,"In the internal shiproom runner notes, what is...",[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0],The rollback plan is considered verified in st...,"[In staging, a verified rollback requires a ti..."
8,qst_0009,basic,"In the EdgePath evaluation email thread, what ...",[dsid_85deb10a652742baaf28af6149600001],Redwood said it wouldn't match the competitor'...,[Redwood did not match the competitors 50 perc...
9,qst_0010,basic,How does the new alerting approach group model...,[dsid_c1a6a71323c04c1ba5445aadea340362],It adds a lightweight token_stage_cohort servi...,[The approach adds a lightweight token_stage_c...


## 4. Create Lightweight QA Dataset
This section maps each selected question to its expected source document and prepares the QA dataset.

In [5]:
# Step 14: Get the matching source document for each selected question

# We create a lookup table using doc_id.
# This helps us quickly find a document by its document ID.

documents_lookup = documents_df.set_index("doc_id")


# Step 15: Create rows for our lightweight QA dataset

lightweight_rows = []

for index, question_row in selected_questions_df.iterrows():

    # Each selected question has exactly one expected document ID.
    expected_doc_id = question_row["expected_doc_ids"][0]

    # Find the matching document from documents_df using that expected_doc_id.
    matching_document = documents_lookup.loc[expected_doc_id]

    # Create one QA test row.
    lightweight_rows.append({
        "test_id": f"RAG_TC_{len(lightweight_rows) + 1:03d}",

        # Unique code created by us for easy tracking.
        "input_code": f"EVL-RAG-{len(lightweight_rows) + 1:03d}",

        # Question details from questions dataset.
        "question_id": question_row["question_id"],
        "question_type": question_row["question_type"],
        "question": question_row["question"],

        # Expected source document details.
        "expected_doc_ids": expected_doc_id,
        "expected_doc_title": matching_document["title"],
        "expected_source_type": matching_document["source_type"],
        "expected_doc_content": matching_document["content"],

        # Expected answer details.
        "gold_answer": question_row["gold_answer"],
        "answer_facts": question_row["answer_facts"],

        # These fields will be filled after running the RAG chatbot/model.
        "actual_retrieved_doc_ids": "",
        "actual_output": "",

        # These fields will be filled by our embedding evaluator later.
        "expected_embedding_code": "",
        "actual_embedding_code": "",
        "similarity_score": "",

        # Final QA result fields.
        "faithfulness_check": "Not Checked",
        "hallucination_check": "Not Checked",
        "match_status": "Not Run",
        "remarks": ""
    })


# Step 16: Convert rows into DataFrame

lightweight_rag_df = pd.DataFrame(lightweight_rows)


# Step 17: Preview the lightweight QA dataset

print("Lightweight QA dataset shape:", lightweight_rag_df.shape)

display(lightweight_rag_df.head())

Lightweight QA dataset shape: (10, 20)


,test_id,input_code,question_id,question_type,question,expected_doc_ids,expected_doc_title,expected_source_type,expected_doc_content,gold_answer,answer_facts,actual_retrieved_doc_ids,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,faithfulness_check,hallucination_check,match_status,remarks
0,RAG_TC_001,EVL-RAG-001,qst_0001,basic,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",github,description:\nMotivation: users integrating to...,The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...,,,,,,Not Checked,Not Checked,Not Run,
1,RAG_TC_002,EVL-RAG-002,qst_0002,basic,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,github,description:\nContext: production customers ob...,The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...,,,,,,Not Checked,Not Checked,Not Run,
2,RAG_TC_003,EVL-RAG-003,qst_0003,basic,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,linear,description:\nObjective: Create a deterministi...,The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...,,,,,,Not Checked,Not Checked,Not Run,
3,RAG_TC_004,EVL-RAG-004,qst_0004,basic,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,fireflies,summary:\nRedwood and the GCP Marketplace team...,The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...,,,,,,Not Checked,Not Checked,Not Run,
4,RAG_TC_005,EVL-RAG-005,qst_0005,basic,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,gmail,['From: Rafael Mendes <rafael.mendes@redwoodin...,MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...,,,,,,Not Checked,Not Checked,Not Run,


In [6]:
# Step 18: Save the lightweight QA dataset as a CSV file

# This CSV is our final ETL output.
# We will use this file for QA testing and embedding comparison later.

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset.csv", index=False)


# Step 19: Confirm the file is saved

print("CSV file created successfully: lightweight_rag_qa_dataset.csv")

CSV file created successfully: lightweight_rag_qa_dataset.csv


Embedding evaluator part

## 5. Generate Expected Embedding Fingerprint Codes
This section converts gold answers into expected embedding fingerprint codes.

In [7]:
# Step 20: Install sentence-transformers
# This library gives us an embedding model.
# Embedding means converting text into numeric meaning code/vector.

!pip install sentence-transformers -q

In [8]:
# Step 21: Load embedding model

# SentenceTransformer is used to convert text into embedding vectors.
from sentence_transformers import SentenceTransformer

# This is a small and commonly used embedding model.
# It converts sentence meaning into a numeric vector.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [9]:
# Step 22: Create expected embedding fingerprint code from gold_answer

# hashlib is used to create a small unique fingerprint from the full embedding.
# This makes the code short and readable, while still being based on the full embedding.
import hashlib
import numpy as np


def create_embedding_fingerprint(text, prefix="EXP"):
    # If text is empty or missing, return empty code.
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # Convert the full text meaning into an embedding vector.
    # This vector represents the meaning of the expected answer.
    embedding_vector = embedding_model.encode(str(text))

    # Round the full embedding vector slightly before hashing.
    # This keeps the fingerprint stable and avoids tiny decimal noise.
    rounded_vector = np.round(embedding_vector, 6)

    # Convert the full rounded vector into bytes.
    # Hashing needs byte format, so we convert the numeric vector into bytes.
    vector_bytes = rounded_vector.tobytes()

    # Create a short hash/fingerprint from the full embedding vector.
    # We use only the first 8 characters to keep the code simple.
    fingerprint = hashlib.sha256(vector_bytes).hexdigest()[:8]

    # Return a small readable embedding code.
    # Example: EXP-a91f23c8
    return f"{prefix}-{fingerprint}"


# Apply the embedding fingerprint generator to gold_answer.
# This creates a short expected embedding code based on the full expected answer meaning.
lightweight_rag_df["expected_embedding_code"] = lightweight_rag_df["gold_answer"].apply(
    lambda text: create_embedding_fingerprint(text, prefix="EXP")
)


# Show the generated expected embedding codes.
display(
    lightweight_rag_df[
        ["test_id", "gold_answer", "expected_embedding_code"]
    ].head()
)


,test_id,gold_answer,expected_embedding_code
0,RAG_TC_001,The default limits are 10 MiB per file (max_fi...,EXP-5bf8dde7
1,RAG_TC_002,The new metric is `stream.timebox_finalized` (...,EXP-510a7333
2,RAG_TC_003,The acceptance criteria are: (1) deliver a sta...,EXP-0451817f
3,RAG_TC_004,The GCP team said entitlement propagation dela...,EXP-7cc8eae3
4,RAG_TC_005,MedThink's preferred failover hierarchy is EU ...,EXP-7eaeb533


In [10]:
# Step 23: Save updated CSV with expected embedding codes

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset_with_expected_codes.csv", index=False)

print("Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv")

Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv


 Developer/QA note:
 Before running the comparison step, paste or load the RAG model outputs into the actual_output column.
 actual_output should contain the answer returned by the RAG chatbot/model for each question.

## 6. Prepare for Actual Output Comparison
This section loads the prepared CSV and checks whether actual model outputs are available.

In [11]:
from google.colab import files

uploaded = files.upload()

Saving lightweight_rag_qa_dataset_with_expected_codes (5).csv to lightweight_rag_qa_dataset_with_expected_codes (5).csv


In [12]:
import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")

print("CSV loaded successfully")
print(qa_df.shape)

CSV loaded successfully
(10, 20)


In [13]:

# Step 24: Load the updated CSV for actual-output comparison stage

# This CSV already contains:
# - selected RAG test questions
# - expected source documents
# - gold_answer
# - expected_embedding_code
# - empty fields for actual_output and comparison results

import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")


# Step 25: Check current QA dataset status

# Total number of test cases available in the CSV.
total_test_cases = len(qa_df)

# Count rows where actual_output is empty.
# actual_output must be filled only after running the RAG model/chatbot.
empty_actual_output_count = qa_df["actual_output"].isna().sum() + (
    qa_df["actual_output"].astype(str).str.strip() == ""
).sum()

# Count rows where expected_embedding_code is already created.
expected_code_count = (
    qa_df["expected_embedding_code"].notna()
    & (qa_df["expected_embedding_code"].astype(str).str.strip() != "")
).sum()


print("Total test cases:", total_test_cases)
print("Rows with expected embedding code:", expected_code_count)
print("Rows waiting for actual model output:", empty_actual_output_count)


# Step 26: Display important QA tracking columns

# This preview helps us verify that:
# - expected answer code is already available
# - actual_output is still waiting for model response
# - result columns are ready for the next comparison phase

display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "expected_embedding_code",
            "actual_output",
            "actual_embedding_code",
            "similarity_score",
            "match_status"
        ]
    ].head()
)


Total test cases: 10
Rows with expected embedding code: 10
Rows waiting for actual model output: 10


,test_id,question,gold_answer,expected_embedding_code,actual_output,actual_embedding_code,similarity_score,match_status
0,RAG_TC_001,What are the default size limits for file uplo...,The default limits are 10 MiB per file (max_fi...,EXP-5bf8dde7,NaN,NaN,NaN,Not Run
1,RAG_TC_002,What is the name of the new metric added so SR...,The new metric is `stream.timebox_finalized` (...,EXP-510a7333,NaN,NaN,NaN,Not Run
2,RAG_TC_003,What are the acceptance criteria for the proje...,The acceptance criteria are: (1) deliver a sta...,EXP-0451817f,NaN,NaN,NaN,Not Run
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,The GCP team said entitlement propagation dela...,EXP-7cc8eae3,NaN,NaN,NaN,Not Run
4,RAG_TC_005,What failover sequence and recovery targets di...,MedThink's preferred failover hierarchy is EU ...,EXP-7eaeb533,NaN,NaN,NaN,Not Run


In [14]:
# Convert these columns into text type before storing text values.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")

## Demo RAG Pipeline Implementation Plan

This section creates a simple demo RAG pipeline only for testing the QA evaluation workflow.

Implementation flow:

1. Create a `document_store` folder.
2. Export each expected document from the CSV into a separate `.txt` file inside `document_store`.
3. Add the document file path back into the dataset as `expected_doc_path`.
4. Add correct documents and later add distractor/wrong documents into the same `document_store`.
5. Demo RAG will read documents from the `document_store` folder.
6. Retrieval will use hybrid search:
   - keyword matching score
   - embedding similarity score
7. The best matching document will be selected as `actual_retrieved_doc_ids`.
8. The answer will be created by extracting the most relevant sentences from the selected document.
9. The generated answer will be stored as `actual_output`.
10. Existing QA checks will run:
    - retrieval match
    - semantic similarity
    - fact-level meaning check
    - overall QA status

Note:

This is not a production RAG chatbot. It is a QA test setup created to generate actual outputs and validate the RAG evaluation workflow end-to-end.

In [15]:
# Step 27: Create document_store folder and export expected documents as .txt files

# This step separates document content from the CSV.
# CSV will keep QA test case details.
# document_store folder will keep source documents like a real RAG knowledge base.

import os
import re

# Create document_store folder if it does not already exist
document_store_path = "document_store"
os.makedirs(document_store_path, exist_ok=True)


def clean_filename(text):
    """
    Create a safe file name from document id/title.
    This removes characters that are not safe for file names.
    """
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_-]", "_", text)
    return text[:120]


# Create a new column to store each exported document file path
qa_df["expected_doc_path"] = ""

# Export each expected document content into a separate text file
for index, row in qa_df.iterrows():
    test_id = row["test_id"]
    doc_id = row["expected_doc_ids"]
    doc_title = row["expected_doc_title"]
    doc_content = row["expected_doc_content"]

    # Create readable and unique file name
    file_name = f"{test_id}_{clean_filename(doc_id)}.txt"
    file_path = os.path.join(document_store_path, file_name)

    # Write document content into .txt file
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(f"doc_id: {doc_id}\n")
        file.write(f"title: {doc_title}\n\n")
        file.write(str(doc_content))

    # Store file path back into dataframe
    qa_df.loc[index, "expected_doc_path"] = file_path


print("Document store created successfully.")
print("Total documents exported:", len(qa_df))
print("Folder name:", document_store_path)

display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "expected_doc_title",
            "expected_doc_path"
        ]
    ]
)

Document store created successfully.
Total documents exported: 10
Folder name: document_store


,test_id,expected_doc_ids,expected_doc_title,expected_doc_path
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",document_store/RAG_TC_001_dsid_ae068ee4aa96401...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,document_store/RAG_TC_002_dsid_9550250a59e74f1...
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,document_store/RAG_TC_003_dsid_3fd6af404fae48e...
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,document_store/RAG_TC_004_dsid_6c4c1c875e704f0...
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,document_store/RAG_TC_005_dsid_8e838ab6a98f4cb...
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,Draft Spec: Policy Engine Extensions for Regio...,document_store/RAG_TC_006_dsid_184be937d34a412...
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,rolling-bias-bisection-log-jared,document_store/RAG_TC_007_dsid_72ec4a9962ba43e...
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Shiproom runner: personal prep log and sticky ...,document_store/RAG_TC_008_dsid_5fc2dba9f6ac4af...
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,Licensing offsets & packaging for potential mi...,document_store/RAG_TC_009_dsid_85deb10a652742b...
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,introduce-token-stage-cohorting-and-route-matr...,document_store/RAG_TC_010_dsid_c1a6a71323c04c1...


In [16]:
# Step 28: Check exported documents inside document_store folder

# This confirms that our document files are created properly.
# The demo RAG system will read documents from this folder.

exported_files = os.listdir(document_store_path)

print("Total files in document_store:", len(exported_files))
print("First few files:")

for file_name in exported_files[:5]:
    print(file_name)

Total files in document_store: 10
First few files:
RAG_TC_006_dsid_184be937d34a412ab5e61366d54d8ed6.txt
RAG_TC_004_dsid_6c4c1c875e704f09b4d791d64d7bc7e5.txt
RAG_TC_008_dsid_5fc2dba9f6ac4af2b49b4f546a4298d0.txt
RAG_TC_007_dsid_72ec4a9962ba43e88acd61abbba1052d.txt
RAG_TC_010_dsid_c1a6a71323c04c1ba5445aadea340362.txt


In [17]:
# Step 29: Load documents from document_store folder

# This step reads the exported .txt files.
# Now the demo RAG system will use file-based documents instead of reading document content directly from CSV.

document_store = []

for file_name in os.listdir(document_store_path):
    if file_name.endswith(".txt"):
        file_path = os.path.join(document_store_path, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        document_store.append(
            {
                "file_name": file_name,
                "file_path": file_path,
                "content": content
            }
        )

print("Documents loaded from document_store:", len(document_store))

# Show one loaded document sample
print("Sample file:", document_store[0]["file_name"])
print("Sample content preview:")
print(document_store[0]["content"][:500])

Documents loaded from document_store: 10
Sample file: RAG_TC_006_dsid_184be937d34a412ab5e61366d54d8ed6.txt
Sample content preview:
doc_id: dsid_184be937d34a412ab5e61366d54d8ed6
title: Draft Spec: Policy Engine Extensions for Regional Failover Automation

Context / why this exists
- We’re upgrading regional failover automation as part of Reliability Program (Q1–Q2 2026). Today, routing policy evaluation has a few gaps:
  - Non-deterministic behavior in edge cases (e.g., multiple signals firing, partial signal availability, time-window ambiguity).
  - Hard to reason about priority across trigger signals (reachability, 5xx bur


In [18]:

# Step 30: Create hybrid retrieval logic

# Hybrid retrieval means:
# 1. Keyword score checks common words between question and document.
# 2. Embedding score checks meaning similarity between question and document.
# 3. Final score combines both.

from sentence_transformers import util
import numpy as np


def keyword_score(question, document_text):
    """
    Simple keyword matching score.
    It checks how many important question words are present in the document.
    """
    question_words = set(str(question).lower().split())
    document_words = set(str(document_text).lower().split())

    if len(question_words) == 0:
        return 0

    matched_words = question_words.intersection(document_words)
    return len(matched_words) / len(question_words)


def hybrid_retrieve(question, document_store):
    """
    Finds the best matching document for a question using hybrid search.
    """

    question_embedding = embedding_model.encode(question, convert_to_tensor=True)

    retrieval_results = []

    for doc in document_store:
        document_text = doc["content"]

        # Keyword match score
        kw_score = keyword_score(question, document_text)

        # Embedding similarity score
        document_embedding = embedding_model.encode(document_text, convert_to_tensor=True)
        emb_score = float(util.cos_sim(question_embedding, document_embedding)[0][0])

        # Final hybrid score
        final_score = (0.4 * kw_score) + (0.6 * emb_score)

        retrieval_results.append(
            {
                "file_name": doc["file_name"],
                "file_path": doc["file_path"],
                "content": document_text,
                "keyword_score": round(kw_score, 4),
                "embedding_score": round(emb_score, 4),
                "hybrid_score": round(final_score, 4)
            }
        )

    # Sort documents by highest hybrid score
    retrieval_results = sorted(
        retrieval_results,
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return retrieval_results[0], retrieval_results


# Test retrieval for first question
sample_question = qa_df.loc[0, "question"]

best_document, all_results = hybrid_retrieve(sample_question, document_store)

print("Question:", sample_question)
print("Best retrieved document:", best_document["file_name"])
print("Hybrid score:", best_document["hybrid_score"])


Question: What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?
Best retrieved document: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Hybrid score: 0.6991


In [19]:
# Step 31: Generate clean demo RAG response from retrieved document

# This step creates a clean actual_output from the retrieved document.
#
# Existing flow is kept:
# question -> retrieve best document -> generate clean response from that document
#
# Output format:
# Answer: <short direct answer>
# Source: <retrieved document id>

!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the retrieved document content.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


def clean_document_text_for_answer_generation(text):
    """
    Creates a cleaned temporary copy of retrieved document text.
    Original document content is not changed.
    """
    lines = str(text).splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if line.startswith("doc_id:"):
            continue

        if line.startswith("title:"):
            continue

        if line == "":
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)


# Load local instruction/text generation model.
# This model generates a clean answer from question + retrieved document.
#rag_response_model_name = "google/flan-t5-base"
rag_response_model_name = "google/flan-t5-large"

rag_tokenizer = AutoTokenizer.from_pretrained(rag_response_model_name)
rag_response_model = AutoModelForSeq2SeqLM.from_pretrained(rag_response_model_name)


def extract_relevant_answer(question, document_text, top_n=3):
    """
    Generates a clean demo RAG response from the retrieved document.

    Input:
    - question: user question
    - document_text: retrieved document content
    - top_n: kept only for Step 32 compatibility

    Output:
    - clean answer with source document id
    """

    doc_id = extract_doc_id_from_file_content(document_text)
    cleaned_document = clean_document_text_for_answer_generation(document_text)

    # Keep context limited so local model can handle it.
    context = cleaned_document[:2500]

    prompt = f"""
    You are a RAG assistant.

    Use the context to answer the question.
    Write the answer in 1 to 3 complete sentences.
    Include all important numbers, names, steps, or conditions needed to answer the question.
    Do not answer with only one word or a short phrase.
    Do not include unrelated context.

    Question:
    {question}

    Context:
    {context}

    Final Answer:
    """

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    with torch.no_grad():
        outputs = rag_response_model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            early_stopping=True
        )

    answer = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if answer == "":
        answer = "Answer not found in retrieved document."

    return f"Answer: {answer}\nSource: {doc_id}"


# Test clean RAG response for first question.
demo_answer = extract_relevant_answer(
    sample_question,
    best_document["content"],
    top_n=3
)

print("Generated demo answer:")
print(demo_answer)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated demo answer:
Answer: 10MiB, 50MiB, 20.
Source: dsid_ae068ee4aa9640159427cd941bef0238


In [20]:
# Step 32: Generate actual retrieved document and actual output for all test cases

# This step runs the demo RAG pipeline for every question.
# It fills:
# - actual_retrieved_doc_ids
# - actual_output
# - actual_embedding_code

def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the first line of exported document file.
    Example line: doc_id: dsid_xxxxx
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


for index, row in qa_df.iterrows():
    question = row["question"]

    # Retrieve best matching document from document_store
    best_document, retrieval_results = hybrid_retrieve(question, document_store)

    # Extract actual retrieved document id
    actual_doc_id = extract_doc_id_from_file_content(best_document["content"])

    # Generate answer from selected document
    generated_answer = extract_relevant_answer(
        question,
        best_document["content"],
        top_n=3
    )

    # Store demo RAG results
    qa_df.loc[index, "actual_retrieved_doc_ids"] = actual_doc_id
    qa_df.loc[index, "actual_output"] = generated_answer

    # Create actual embedding fingerprint code
    qa_df.loc[index, "actual_embedding_code"] = create_embedding_fingerprint(
        generated_answer,
        prefix="ACT"
    )


# Display updated output
display(
    qa_df[
        [
            "test_id",
            "question",
            "expected_doc_ids",
            "actual_retrieved_doc_ids",
            "actual_output",
            "actual_embedding_code"
        ]
    ]
)

/tmp/ipykernel_4441/4146286004.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'dsid_ae068ee4aa9640159427cd941bef0238' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  qa_df.loc[index, "actual_retrieved_doc_ids"] = actual_doc_id


,test_id,question,expected_doc_ids,actual_retrieved_doc_ids,actual_output,actual_embedding_code
0,RAG_TC_001,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,dsid_ae068ee4aa9640159427cd941bef0238,"Answer: 10MiB, 50MiB, 20.\nSource: dsid_ae068e...",ACT-161feb38
1,RAG_TC_002,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,dsid_9550250a59e74f1bbd5612480b2e7100,Answer: Timeboxed.\nSource: dsid_9550250a59e74...,ACT-856e5ecc
2,RAG_TC_003,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Answer: Implementation should use runtime CSS ...,ACT-30db4369
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,Answer: GCP emphasized keeping dimension names...,ACT-e95202f1
4,RAG_TC_005,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,dsid_184be937d34a412ab5e61366d54d8ed6,"Answer: No-op, failover, partial shift, rollba...",ACT-a246890b
5,RAG_TC_006,In the draft spec about extending a routing po...,dsid_184be937d34a412ab5e61366d54d8ed6,dsid_184be937d34a412ab5e61366d54d8ed6,Answer: Reachability: ok|degraded|down + confi...,ACT-19411e24
6,RAG_TC_007,In a rolling investigation of a model regressi...,dsid_72ec4a9962ba43e88acd61abbba1052d,dsid_72ec4a9962ba43e88acd61abbba1052d,Answer: The average triage rubric score change...,ACT-1ca6c75b
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Answer: 10m.\nSource: dsid_5fc2dba9f6ac4af2b49...,ACT-70b669e5
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",dsid_85deb10a652742baaf28af6149600001,dsid_85deb10a652742baaf28af6149600001,Answer: Prepay bucket + seat licenses with an ...,ACT-a28081b1
9,RAG_TC_010,How does the new alerting approach group model...,dsid_c1a6a71323c04c1ba5445aadea340362,dsid_c1a6a71323c04c1ba5445aadea340362,Answer: Introduces an alert route-matrix gener...,ACT-2e04f80f


In [21]:
# Step 33: Save dataset with demo RAG generated outputs

# This CSV now contains:
# - expected answers
# - expected document IDs
# - actual retrieved document IDs from demo RAG
# - actual outputs from demo RAG
# - actual embedding codes

demo_rag_output_file = "rag_qa_dataset_with_demo_rag_outputs.csv"

qa_df.to_csv(demo_rag_output_file, index=False)

print("Demo RAG output CSV saved successfully:", demo_rag_output_file)

Demo RAG output CSV saved successfully: rag_qa_dataset_with_demo_rag_outputs.csv


## 8. Retrieval Document Check
This section checks whether the RAG system retrieved the expected source document.

In [22]:
# Step 28: Retrieval document check

# This logic checks whether the RAG system retrieved the correct source document.
# It works for all rows in the dataset.
# If actual_retrieved_doc_ids is empty, the row is marked as Not Run.

# Create result columns for retrieval evaluation.
qa_df["retrieval_match_status"] = ""
qa_df["retrieval_remarks"] = ""


def normalize_doc_ids(value):
    # Convert document ID values into a clean list.
    # This handles:
    # - empty values
    # - single document ID as text
    # - list-like document IDs saved as text

    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    # Remove brackets and quotes if IDs are stored like ['doc_id']
    value = value.replace("[", "").replace("]", "").replace("'", "").replace('"', "")

    # Split by comma in case multiple document IDs are present.
    doc_ids = [item.strip() for item in value.split(",") if item.strip()]

    return doc_ids


# Run retrieval check for every row.
for row_index, row in qa_df.iterrows():

    expected_doc_ids = normalize_doc_ids(row["expected_doc_ids"])
    actual_doc_ids = normalize_doc_ids(row["actual_retrieved_doc_ids"])

    # If actual retrieved document is not available, we cannot run retrieval check yet.
    if len(actual_doc_ids) == 0:
        qa_df.loc[row_index, "retrieval_match_status"] = "Not Run"
        qa_df.loc[row_index, "retrieval_remarks"] = "Actual retrieved document ID is not available yet."
        continue

    # Check whether any expected document ID is present in actual retrieved document IDs.
    is_match = any(doc_id in actual_doc_ids for doc_id in expected_doc_ids)

    if is_match:
        qa_df.loc[row_index, "retrieval_match_status"] = "Pass"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was retrieved."
    else:
        qa_df.loc[row_index, "retrieval_match_status"] = "Fail"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was not retrieved."


# Show retrieval check result.
display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "actual_retrieved_doc_ids",
            "retrieval_match_status",
            "retrieval_remarks"
        ]
    ]
)

,test_id,expected_doc_ids,actual_retrieved_doc_ids,retrieval_match_status,retrieval_remarks
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,dsid_ae068ee4aa9640159427cd941bef0238,Pass,Expected document was retrieved.
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,dsid_9550250a59e74f1bbd5612480b2e7100,Pass,Expected document was retrieved.
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Pass,Expected document was retrieved.
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,Pass,Expected document was retrieved.
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,dsid_184be937d34a412ab5e61366d54d8ed6,Fail,Expected document was not retrieved.
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,dsid_184be937d34a412ab5e61366d54d8ed6,Pass,Expected document was retrieved.
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,dsid_72ec4a9962ba43e88acd61abbba1052d,Pass,Expected document was retrieved.
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Pass,Expected document was retrieved.
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,dsid_85deb10a652742baaf28af6149600001,Pass,Expected document was retrieved.
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,dsid_c1a6a71323c04c1ba5445aadea340362,Pass,Expected document was retrieved.


In [23]:
## 9. Context Precision Check

# This section checks whether the retrieved document/context is useful
# for answering the question.
#
# Simple meaning:
# - Retrieval check only checks whether the expected document ID was retrieved.
# - Context Precision checks whether the retrieved document content is useful.
#
# Correct comparison:
# gold_answer  vs  retrieved document content


def get_retrieved_document_content(retrieved_doc_id, document_store):
    """
    Finds the retrieved document content using actual_retrieved_doc_ids.

    Here document_store is a list of documents loaded from the document_store folder.
    """

    if pd.isna(retrieved_doc_id) or str(retrieved_doc_id).strip() == "":
        return ""

    retrieved_doc_id = str(retrieved_doc_id).strip()

    # Search each loaded document and check whether the doc_id is present inside it.
    for document in document_store:
        content = str(document["content"])

        if f"doc_id: {retrieved_doc_id}" in content:
            return content

    return ""

def context_precision_check(reference_answer, retrieved_context, threshold=0.60):
    """
    Compares expected answer meaning with retrieved document content.

    Input:
    - reference_answer: gold_answer / expected answer
    - retrieved_context: retrieved document content

    Output:
    - context precision score
    - Pass/Fail status
    - remarks
    """

    if pd.isna(reference_answer) or str(reference_answer).strip() == "":
        return "", "Not Run", "Reference answer is missing."

    if pd.isna(retrieved_context) or str(retrieved_context).strip() == "":
        return "", "Not Run", "Retrieved document content is missing."

    # Convert expected answer into embedding.
    reference_embedding = embedding_model.encode(
        str(reference_answer),
        convert_to_tensor=True
    )

    # Convert retrieved document content into embedding.
    context_embedding = embedding_model.encode(
        str(retrieved_context),
        convert_to_tensor=True
    )

    # Compare meaning similarity between expected answer and retrieved document content.
    score = float(util.cos_sim(reference_embedding, context_embedding)[0][0])
    score = round(score, 4)

    if score >= threshold:
        return score, "Pass", "Retrieved document content is useful for answering the question."

    return score, "Fail", "Retrieved document content is not useful enough for answering the question."


# Create columns for storing retrieved document content and Context Precision result.
qa_df["actual_retrieved_doc_content"] = ""
qa_df["context_precision_score"] = ""
qa_df["context_precision_status"] = ""
qa_df["context_precision_remarks"] = ""


# Run Context Precision check for every row.
for row_index, row in qa_df.iterrows():

    # Step 1: Get the retrieved document content using actual_retrieved_doc_ids.

    retrieved_context = get_retrieved_document_content(
        retrieved_doc_id=row["actual_retrieved_doc_ids"],
        document_store=document_store
    )

    # Step 2: Store retrieved document content for QA traceability.
    qa_df.loc[row_index, "actual_retrieved_doc_content"] = retrieved_context

    # Step 3: Compare gold_answer with retrieved document content.
    score, status, remarks = context_precision_check(
        reference_answer=row["gold_answer"],
        retrieved_context=retrieved_context
    )

    # Step 4: Store Context Precision result.
    qa_df.loc[row_index, "context_precision_score"] = score
    qa_df.loc[row_index, "context_precision_status"] = status
    qa_df.loc[row_index, "context_precision_remarks"] = remarks


# Show Context Precision result.
display(
    qa_df[
        [
            "test_id",
            "gold_answer",
            "actual_retrieved_doc_ids",
            "context_precision_score",
            "context_precision_status",
            "context_precision_remarks"
        ]
    ]
)

,test_id,gold_answer,actual_retrieved_doc_ids,context_precision_score,context_precision_status,context_precision_remarks
0,RAG_TC_001,The default limits are 10 MiB per file (max_fi...,dsid_ae068ee4aa9640159427cd941bef0238,0.6742,Pass,Retrieved document content is useful for answe...
1,RAG_TC_002,The new metric is `stream.timebox_finalized` (...,dsid_9550250a59e74f1bbd5612480b2e7100,0.4428,Fail,Retrieved document content is not useful enoug...
2,RAG_TC_003,The acceptance criteria are: (1) deliver a sta...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,0.598,Fail,Retrieved document content is not useful enoug...
3,RAG_TC_004,The GCP team said entitlement propagation dela...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,0.4625,Fail,Retrieved document content is not useful enoug...
4,RAG_TC_005,MedThink's preferred failover hierarchy is EU ...,dsid_184be937d34a412ab5e61366d54d8ed6,0.4845,Fail,Retrieved document content is not useful enoug...
5,RAG_TC_006,The draft proposes this canonical v1 signal pr...,dsid_184be937d34a412ab5e61366d54d8ed6,0.5534,Fail,Retrieved document content is not useful enoug...
6,RAG_TC_007,"In the first comparison run, the baseline buil...",dsid_72ec4a9962ba43e88acd61abbba1052d,0.2369,Fail,Retrieved document content is not useful enoug...
7,RAG_TC_008,The rollback plan is considered verified in st...,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,0.1982,Fail,Retrieved document content is not useful enoug...
8,RAG_TC_009,Redwood said it wouldn't match the competitor'...,dsid_85deb10a652742baaf28af6149600001,0.6017,Pass,Retrieved document content is useful for answe...
9,RAG_TC_010,It adds a lightweight token_stage_cohort servi...,dsid_c1a6a71323c04c1ba5445aadea340362,0.8506,Pass,Retrieved document content is useful for answe...


In [24]:
## 10. Response Relevancy Check

# This section checks whether actual_output is relevant to the original question.
#
# Ragas-style logic:
# 1. Take actual_output.
# 2. Use a local Hugging Face question-generation model to generate questions.
# 3. Convert generated questions and original question into embeddings.
# 4. Compare them using cosine similarity.
# 5. Average the score and mark Pass/Fail.
#
# This version does not use OpenAI API or Gemini API.
# It runs locally inside Colab after downloading the Hugging Face model.

!pip install transformers sentencepiece -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np


# Load local Hugging Face question-generation model.
# This model is used to generate questions from actual_output.
question_generation_model_name = "valhalla/t5-small-qg-hl"

qg_tokenizer = AutoTokenizer.from_pretrained(question_generation_model_name)
qg_model = AutoModelForSeq2SeqLM.from_pretrained(question_generation_model_name)


def generate_questions_from_answer(actual_output, question_count=3):
    """
    Generates possible questions from actual_output using a local Hugging Face model.

    Input:
    - actual_output: answer generated by the demo RAG pipeline
    - question_count: number of questions to generate

    Output:
    - list of generated questions
    """

    if pd.isna(actual_output) or str(actual_output).strip() == "":
        return []

    # Keep the text short enough for the model.
    #answer_text = str(actual_output).strip()[:1000]
    answer_text = str(actual_output).strip()

    # If actual_output has Answer and Source, use only Answer part.
    if "Source:" in answer_text:
        answer_text = answer_text.split("Source:")[0].strip()

    # Remove Answer label before sending to question generator.
    answer_text = answer_text.replace("Answer:", "").strip()

    # Keep text short enough for the question generation model.
    answer_text = answer_text[:1000]

    # Prompt format for question generation.
    input_text = "generate question: " + answer_text

    inputs = qg_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    outputs = qg_model.generate(
        **inputs,
        max_length=64,
        num_return_sequences=question_count,
        num_beams=5,
        do_sample=True
    )

    generated_questions = [
        qg_tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

    return generated_questions


def response_relevancy_check(original_question, actual_output, threshold=0.60):
    """
    Checks whether actual_output is relevant to the original question.

    Input:
    - original_question: original question from dataset
    - actual_output: answer generated by demo RAG pipeline

    Output:
    - generated questions
    - response relevancy score
    - Pass/Fail status
    - remarks
    """

    if pd.isna(original_question) or str(original_question).strip() == "":
        return [], "", "Not Run", "Original question is missing."

    if pd.isna(actual_output) or str(actual_output).strip() == "":
        return [], "", "Not Run", "Actual output is missing."

    # Step 1: Generate questions from actual_output.
    generated_questions = generate_questions_from_answer(actual_output)

    if len(generated_questions) == 0:
        return [], "", "Not Run", "Question generation failed."

    # Step 2: Convert original question into embedding.
    original_question_embedding = embedding_model.encode(
        str(original_question),
        convert_to_tensor=True
    )

    similarity_scores = []

    # Step 3: Compare each generated question with original question.
    for generated_question in generated_questions:

        generated_question_embedding = embedding_model.encode(
            str(generated_question),
            convert_to_tensor=True
        )

        score = float(
            util.cos_sim(
                original_question_embedding,
                generated_question_embedding
            )[0][0]
        )

        similarity_scores.append(score)

    # Step 4: Average all similarity scores.
    final_score = round(float(np.mean(similarity_scores)), 4)

    if final_score >= threshold:
        return generated_questions, final_score, "Pass", "Actual output is relevant to the question."

    return generated_questions, final_score, "Fail", "Actual output is not relevant enough to the question."


# Create columns for Response Relevancy result.
qa_df["generated_question_from_answer"] = ""
qa_df["response_relevancy_score"] = ""
qa_df["response_relevancy_status"] = ""
qa_df["response_relevancy_remarks"] = ""


# Run Response Relevancy check for every row.
for row_index, row in qa_df.iterrows():

    generated_questions, score, status, remarks = response_relevancy_check(
        original_question=row["question"],
        actual_output=row["actual_output"]
    )

    qa_df.loc[row_index, "generated_question_from_answer"] = str(generated_questions)
    qa_df.loc[row_index, "response_relevancy_score"] = score
    qa_df.loc[row_index, "response_relevancy_status"] = status
    qa_df.loc[row_index, "response_relevancy_remarks"] = remarks


# Show Response Relevancy result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "actual_output",
            "generated_question_from_answer",
            "response_relevancy_score",
            "response_relevancy_status",
            "response_relevancy_remarks"
        ]
    ]
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

,test_id,question,actual_output,generated_question_from_answer,response_relevancy_score,response_relevancy_status,response_relevancy_remarks
0,RAG_TC_001,What are the default size limits for file uplo...,"Answer: 10MiB, 50MiB, 20.\nSource: dsid_ae068e...","['What are 10MiB, 50MiB, 20?', 'What are 10MiB...",0.205,Fail,Actual output is not relevant enough to the qu...
1,RAG_TC_002,What is the name of the new metric added so SR...,Answer: Timeboxed.\nSource: dsid_9550250a59e74...,"['What does Timeboxed do?', 'What does Timebox...",0.2918,Fail,Actual output is not relevant enough to the qu...
2,RAG_TC_003,What are the acceptance criteria for the proje...,Answer: Implementation should use runtime CSS ...,['What should implementation use runtime CSS v...,0.1882,Fail,Actual output is not relevant enough to the qu...
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,Answer: GCP emphasized keeping dimension names...,['What did GCP emphasis on keeping dimension n...,0.5737,Fail,Actual output is not relevant enough to the qu...
4,RAG_TC_005,What failover sequence and recovery targets di...,"Answer: No-op, failover, partial shift, rollba...","['What do controllers emit?', 'What is the nam...",-0.0533,Fail,Actual output is not relevant enough to the qu...
5,RAG_TC_006,In the draft spec about extending a routing po...,Answer: Reachability: ok|degraded|down + confi...,['What is ok|degraded|down + confidence + last...,0.2535,Fail,Actual output is not relevant enough to the qu...
6,RAG_TC_007,In a rolling investigation of a model regressi...,Answer: The average triage rubric score change...,['What was the average triage rubric score cha...,0.68,Pass,Actual output is relevant to the question.
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",Answer: 10m.\nSource: dsid_5fc2dba9f6ac4af2b49...,"['What is the height of 10m?', 'What is the he...",0.1333,Fail,Actual output is not relevant enough to the qu...
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",Answer: Prepay bucket + seat licenses with an ...,['What do prepay bucket + seat licenses with a...,0.2318,Fail,Actual output is not relevant enough to the qu...
9,RAG_TC_010,How does the new alerting approach group model...,Answer: Introduces an alert route-matrix gener...,['What is an alert route-matrix generator that...,0.5418,Fail,Actual output is not relevant enough to the qu...


## 9. Semantic Similarity Comparison
This section compares expected answers and actual outputs using embedding similarity score.

In [25]:
# Step 28: Compare expected answers and actual outputs using embeddings

# This comparison logic is built for the full dataset.
# It will run only for rows where actual_output is available.
# Rows without actual_output will remain as "Not Run".

import numpy as np


# Make sure these columns can store text/results properly.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")
qa_df["similarity_score"] = qa_df["similarity_score"].astype("object")
qa_df["match_status"] = qa_df["match_status"].astype("object")
qa_df["remarks"] = qa_df["remarks"].astype("object")


# Similarity threshold.
# If score is 0.80 or above, we mark it as Pass.
# You can tune this later based on project requirement.
SIMILARITY_THRESHOLD = 0.80


def calculate_similarity(expected_text, actual_text):
    # Convert expected answer into embedding vector.
    expected_vector = embedding_model.encode(str(expected_text))

    # Convert actual answer into embedding vector.
    actual_vector = embedding_model.encode(str(actual_text))

    # Calculate cosine similarity.
    # Higher score means both answers have closer meaning.
    similarity = np.dot(expected_vector, actual_vector) / (
        np.linalg.norm(expected_vector) * np.linalg.norm(actual_vector)
    )

    return round(float(similarity), 4)


# Loop through every test case in the QA dataset.
for row_index, row in qa_df.iterrows():

    # If actual_output is empty, we cannot compare this row yet.
    if pd.isna(row["actual_output"]) or str(row["actual_output"]).strip() == "":
        qa_df.loc[row_index, "match_status"] = "Not Run"
        qa_df.loc[row_index, "remarks"] = "Actual output is not available yet."
        continue

    # Create actual embedding fingerprint code from actual_output.
    qa_df.loc[row_index, "actual_embedding_code"] = create_embedding_fingerprint(
        row["actual_output"],
        prefix="ACT"
    )

    # Calculate similarity between gold_answer and actual_output.
    score = calculate_similarity(
        row["gold_answer"],
        row["actual_output"]
    )

    qa_df.loc[row_index, "similarity_score"] = score

    # Mark Pass/Fail based on similarity threshold.
    if score >= SIMILARITY_THRESHOLD:
        qa_df.loc[row_index, "match_status"] = "Pass"
        qa_df.loc[row_index, "remarks"] = "Actual output is semantically similar to expected answer."
    else:
        qa_df.loc[row_index, "match_status"] = "Fail"
        qa_df.loc[row_index, "remarks"] = "Actual output is not similar enough to expected answer."


# Show final comparison status.
display(
    qa_df[
        [
            "test_id",
            "gold_answer",
            "actual_output",
            "expected_embedding_code",
            "actual_embedding_code",
            "similarity_score",
            "match_status",
            "remarks"
        ]
    ]
)

,test_id,gold_answer,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,match_status,remarks
0,RAG_TC_001,The default limits are 10 MiB per file (max_fi...,"Answer: 10MiB, 50MiB, 20.\nSource: dsid_ae068e...",EXP-5bf8dde7,ACT-161feb38,0.3644,Fail,Actual output is not similar enough to expecte...
1,RAG_TC_002,The new metric is `stream.timebox_finalized` (...,Answer: Timeboxed.\nSource: dsid_9550250a59e74...,EXP-510a7333,ACT-856e5ecc,0.4689,Fail,Actual output is not similar enough to expecte...
2,RAG_TC_003,The acceptance criteria are: (1) deliver a sta...,Answer: Implementation should use runtime CSS ...,EXP-0451817f,ACT-30db4369,0.2262,Fail,Actual output is not similar enough to expecte...
3,RAG_TC_004,The GCP team said entitlement propagation dela...,Answer: GCP emphasized keeping dimension names...,EXP-7cc8eae3,ACT-e95202f1,0.5112,Fail,Actual output is not similar enough to expecte...
4,RAG_TC_005,MedThink's preferred failover hierarchy is EU ...,"Answer: No-op, failover, partial shift, rollba...",EXP-7eaeb533,ACT-a246890b,0.2732,Fail,Actual output is not similar enough to expecte...
5,RAG_TC_006,The draft proposes this canonical v1 signal pr...,Answer: Reachability: ok|degraded|down + confi...,EXP-beea7969,ACT-19411e24,0.4763,Fail,Actual output is not similar enough to expecte...
6,RAG_TC_007,"In the first comparison run, the baseline buil...",Answer: The average triage rubric score change...,EXP-487ff8cd,ACT-1ca6c75b,0.4765,Fail,Actual output is not similar enough to expecte...
7,RAG_TC_008,The rollback plan is considered verified in st...,Answer: 10m.\nSource: dsid_5fc2dba9f6ac4af2b49...,EXP-2d4f9d4d,ACT-70b669e5,0.118,Fail,Actual output is not similar enough to expecte...
8,RAG_TC_009,Redwood said it wouldn't match the competitor'...,Answer: Prepay bucket + seat licenses with an ...,EXP-7fece42e,ACT-a28081b1,0.4555,Fail,Actual output is not similar enough to expecte...
9,RAG_TC_010,It adds a lightweight token_stage_cohort servi...,Answer: Introduces an alert route-matrix gener...,EXP-a4d20d9d,ACT-2e04f80f,0.818,Pass,Actual output is semantically similar to expec...


In [26]:
# Step 29: Save final demo comparison result

# This CSV includes:
# - expected answers
# - expected embedding codes
# - one demo actual output
# - actual embedding code
# - similarity score
# - pass/fail result for the demo row

qa_df.to_csv("rag_qa_demo_comparison_result.csv", index=False)

print("Final demo comparison CSV saved: rag_qa_demo_comparison_result.csv")

Final demo comparison CSV saved: rag_qa_demo_comparison_result.csv


In [27]:
# Download the final CSV from Colab to your computer

from google.colab import files

files.download("rag_qa_demo_comparison_result.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Fact-Level Meaning Check

This section checks whether the expected answer facts are present in the actual output using sentence-level embedding similarity.

## 10. Fact-Level Meaning Check
This section checks whether expected answer facts are present in the actual output using sentence-level embedding similarity.

In [28]:
# Step 30: Fact-level meaning check using embeddings

# This logic checks whether each expected fact is present in the actual output.
# It does not depend on exact same words.
# It compares meaning using embeddings.

import ast
import re
import numpy as np


# Convert result columns into object type so they can store text/list values.
qa_df["matched_facts"] = ""
qa_df["missing_facts"] = ""
qa_df["fact_match_score"] = ""
qa_df["fact_check_status"] = ""


# Threshold for fact-level semantic match.
# If expected fact and one actual sentence are this similar or higher,
# we treat that expected fact as present in the actual output.
FACT_MATCH_THRESHOLD = 0.70


def parse_answer_facts(answer_facts_value):
    # answer_facts may come as a list or as a string version of a list.
    # This function converts it safely into a Python list.

    if isinstance(answer_facts_value, list):
        return answer_facts_value

    if pd.isna(answer_facts_value) or str(answer_facts_value).strip() == "":
        return []

    try:
        parsed_value = ast.literal_eval(str(answer_facts_value))
        if isinstance(parsed_value, list):
            return parsed_value
        return [str(parsed_value)]
    except:
        return [str(answer_facts_value)]


def split_into_sentences(text):
    # Split actual output into smaller sentence-level parts.
    # This helps us check expected fact vs each actual sentence.

    if pd.isna(text) or str(text).strip() == "":
        return []

    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def cosine_similarity(text1, text2):
    # Convert both texts into embeddings and compare their meaning.

    vector1 = embedding_model.encode(str(text1))
    vector2 = embedding_model.encode(str(text2))

    score = np.dot(vector1, vector2) / (
        np.linalg.norm(vector1) * np.linalg.norm(vector2)
    )

    return float(score)


# Run fact-level check for every row.
for row_index, row in qa_df.iterrows():

    # If actual_output is not available, fact check cannot be done.
    if pd.isna(row["actual_output"]) or str(row["actual_output"]).strip() == "":
        qa_df.loc[row_index, "fact_check_status"] = "Not Run"
        qa_df.loc[row_index, "matched_facts"] = ""
        qa_df.loc[row_index, "missing_facts"] = ""
        qa_df.loc[row_index, "fact_match_score"] = ""
        continue

    expected_facts = parse_answer_facts(row["answer_facts"])
    actual_sentences = split_into_sentences(row["actual_output"])

    matched_facts = []
    missing_facts = []

    for expected_fact in expected_facts:
        best_score = 0

        # Compare one expected fact with every actual sentence.
        for actual_sentence in actual_sentences:
            score = cosine_similarity(expected_fact, actual_sentence)
            best_score = max(best_score, score)

        # If best sentence match is above threshold, fact is matched.
        if best_score >= FACT_MATCH_THRESHOLD:
            matched_facts.append(expected_fact)
        else:
            missing_facts.append(expected_fact)

    total_facts = len(expected_facts)
    matched_count = len(matched_facts)

    if total_facts == 0:
        fact_score = ""
        fact_status = "No Expected Facts"
    else:
        fact_score = round(matched_count / total_facts, 4)

        if matched_count == total_facts:
            fact_status = "Pass"
        elif matched_count > 0:
            fact_status = "Partial"
        else:
            fact_status = "Fail"

    qa_df.loc[row_index, "matched_facts"] = str(matched_facts)
    qa_df.loc[row_index, "missing_facts"] = str(missing_facts)
    qa_df.loc[row_index, "fact_match_score"] = fact_score
    qa_df.loc[row_index, "fact_check_status"] = fact_status


# Show fact-level check result.
display(
    qa_df[
        [
            "test_id",
            "actual_output",
            "answer_facts",
            "matched_facts",
            "missing_facts",
            "fact_match_score",
            "fact_check_status"
        ]
    ]
)

,test_id,actual_output,answer_facts,matched_facts,missing_facts,fact_match_score,fact_check_status
0,RAG_TC_001,"Answer: 10MiB, 50MiB, 20.\nSource: dsid_ae068e...",['The default per file upload size limit (max_...,[],['The default per file upload size limit (max_...,0.0,Fail
1,RAG_TC_002,Answer: Timeboxed.\nSource: dsid_9550250a59e74...,['The new metric added is named stream.timebox...,[],['The new metric added is named stream.timebox...,0.0,Fail
2,RAG_TC_003,Answer: Implementation should use runtime CSS ...,['Deliver a stable design JSON token spec that...,[],['Deliver a stable design JSON token spec that...,0.0,Fail
3,RAG_TC_004,Answer: GCP emphasized keeping dimension names...,['Entitlement propagation delays can occur aft...,[],['Entitlement propagation delays can occur aft...,0.0,Fail
4,RAG_TC_005,"Answer: No-op, failover, partial shift, rollba...",['MedThink specified a failover hierarchy of E...,[],['MedThink specified a failover hierarchy of E...,0.0,Fail
5,RAG_TC_006,Answer: Reachability: ok|degraded|down + confi...,['The draft spec proposes a canonical v1 prior...,[],['The draft spec proposes a canonical v1 prior...,0.0,Fail
6,RAG_TC_007,Answer: The average triage rubric score change...,"['In the first comparison run, the older basel...","['In the first comparison run, the older basel...",[],1.0,Pass
7,RAG_TC_008,Answer: 10m.\nSource: dsid_5fc2dba9f6ac4af2b49...,"['In staging, a verified rollback requires a t...",[],"['In staging, a verified rollback requires a t...",0.0,Fail
8,RAG_TC_009,Answer: Prepay bucket + seat licenses with an ...,['Redwood did not match the competitors 50 per...,[],['Redwood did not match the competitors 50 per...,0.0,Fail
9,RAG_TC_010,Answer: Introduces an alert route-matrix gener...,['The approach adds a lightweight token_stage_...,['The approach adds a lightweight token_stage_...,[],1.0,Pass


## Overall QA Status
This section combines retrieval check, semantic similarity check, and fact-level check into one final QA status.

In [29]:
# Overall QA Status

# This logic combines three checks:
# 1. retrieval_match_status -> whether correct document was retrieved
# 2. match_status -> whether actual answer is semantically similar to expected answer
# 3. fact_check_status -> whether expected facts are present in actual answer

qa_df["overall_qa_status"] = ""
qa_df["overall_qa_remarks"] = ""


for row_index, row in qa_df.iterrows():

    retrieval_status = str(row.get("retrieval_match_status", "")).strip()
    similarity_status = str(row.get("match_status", "")).strip()
    fact_status = str(row.get("fact_check_status", "")).strip()

    failed_checks = []
    not_run_checks = []

    # Track checks that are not completed
    if retrieval_status == "" or retrieval_status == "Not Run":
        not_run_checks.append("retrieval check")

    if similarity_status == "" or similarity_status == "Not Run":
        not_run_checks.append("semantic similarity check")

    if fact_status == "" or fact_status == "Not Run":
        not_run_checks.append("fact-level check")

    # If any required check is not run, overall status is Not Run.
    if len(not_run_checks) > 0:
        qa_df.loc[row_index, "overall_qa_status"] = "Not Run"
        qa_df.loc[row_index, "overall_qa_remarks"] = (
            "Not completed: " + ", ".join(not_run_checks)
        )
        continue

    # Track failed checks
    if retrieval_status == "Fail":
        failed_checks.append("retrieval check failed")

    if similarity_status == "Fail":
        failed_checks.append("semantic similarity check failed")

    if fact_status == "Fail":
        failed_checks.append("fact-level check failed")

    # If all checks pass, overall status is Pass.
    if (
        retrieval_status == "Pass"
        and similarity_status == "Pass"
        and fact_status == "Pass"
    ):
        qa_df.loc[row_index, "overall_qa_status"] = "Pass"
        qa_df.loc[row_index, "overall_qa_remarks"] = (
            "Retrieval, semantic similarity, and fact-level checks passed."
        )
        continue

    # If fact check is partial, overall status is Partial.
    if fact_status == "Partial":
        qa_df.loc[row_index, "overall_qa_status"] = "Partial"

        if len(failed_checks) > 0:
            qa_df.loc[row_index, "overall_qa_remarks"] = (
                "; ".join(failed_checks) + "; fact-level check partially passed"
            )
        else:
            qa_df.loc[row_index, "overall_qa_remarks"] = (
                "Fact-level check partially passed; some expected facts are missing."
            )
        continue

    # If any check fails, overall status is Fail.
    if len(failed_checks) > 0:
        qa_df.loc[row_index, "overall_qa_status"] = "Fail"
        qa_df.loc[row_index, "overall_qa_remarks"] = "; ".join(failed_checks)
        continue


# Show final QA status.
display(
    qa_df[
        [
            "test_id",
            "retrieval_match_status",
            "match_status",
            "fact_check_status",
            "overall_qa_status",
            "overall_qa_remarks"
        ]
    ]
)

,test_id,retrieval_match_status,match_status,fact_check_status,overall_qa_status,overall_qa_remarks
0,RAG_TC_001,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
1,RAG_TC_002,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
2,RAG_TC_003,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
3,RAG_TC_004,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
4,RAG_TC_005,Fail,Fail,Fail,Fail,retrieval check failed; semantic similarity ch...
5,RAG_TC_006,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
6,RAG_TC_007,Pass,Fail,Pass,Fail,semantic similarity check failed
7,RAG_TC_008,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
8,RAG_TC_009,Pass,Fail,Fail,Fail,semantic similarity check failed; fact-level c...
9,RAG_TC_010,Pass,Pass,Pass,Pass,"Retrieval, semantic similarity, and fact-level..."


In [30]:
# Save updated result CSV with similarity + fact-level check results

qa_df.to_csv("rag_qa_final_demo_result_with_all_checks.csv", index=False)

print("Saved: rag_qa_final_demo_result_with_all_checks.csv")

Saved: rag_qa_final_demo_result_with_all_checks.csv


In [31]:
import os

print(os.path.exists("rag_qa_final_demo_result_with_all_checks.csv"))
print(os.path.getsize("rag_qa_final_demo_result_with_all_checks.csv"))
# Download updated CSV to your computer

from google.colab import files

files.download("rag_qa_final_demo_result_with_all_checks.csv")

True
208088


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>